# Geneformer Foundation Model vs Scanpy PCA: PBMC 3K

## Project objective

This notebook validates and explores a comparison between pretrained Geneformer V1 cell embeddings and a conventional Scanpy PCA representation.

**Important scope:** this notebook analyses saved, validated outputs. It does not rerun Geneformer tokenization, model inference, or the complete Scanpy preprocessing pipeline from raw counts.

## Dataset and representations

- Dataset: PBMC 3K
- Cells: 2,671
- Supplied cell-type groups: 6
- Geneformer V1: 256-dimensional mean-pooled second-to-last-layer embeddings
- Scanpy baseline: 2,000 highly variable genes and 50 PCA components
- Evaluation metric: cosine silhouette score

Cell-type annotations are preliminary and are not independent biological ground truth.


## 1. Imports and paths

Paths are resolved from the repository root when the notebook is launched there, or from the notebook's parent directory when launched inside `notebooks/`.


In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

cwd = Path.cwd()
if (cwd / "results" / "verified_comparison").exists():
    PROJECT_DIR = cwd
elif (cwd.parent / "results" / "verified_comparison").exists():
    PROJECT_DIR = cwd.parent
else:
    raise FileNotFoundError(
        "Cannot find results/verified_comparison. "
        "Run the notebook from the repository root or notebooks/."
    )

RESULTS_DIR = PROJECT_DIR / "results" / "verified_comparison"

print("Project directory:", PROJECT_DIR.resolve())
print("Comparison results:", RESULTS_DIR.resolve())


## 2. Load the validated comparison outputs

The notebook loads the exported embedding matrices, UMAP coordinates, canonical cell-type summary, and canonical metrics/provenance JSON. It recalculates silhouette metrics from the embeddings using corrected canonical labels.


In [ ]:
embedding_df = pd.read_csv(RESULTS_DIR / "geneformer_embeddings_256d.csv")
pca_df = pd.read_csv(RESULTS_DIR / "scanpy_pca_scores_50d.csv")
canonical_type_summary = pd.read_csv(
    RESULTS_DIR / "cell_type_silhouette_summary_canonical.csv"
)
geneformer_umap = pd.read_csv(RESULTS_DIR / "geneformer_umap_coordinates.csv")
pca_umap = pd.read_csv(RESULTS_DIR / "scanpy_pca_umap_coordinates.csv")

with (RESULTS_DIR / "comparison_metrics.json").open(encoding="utf-8") as f:
    saved_metrics = json.load(f)

print("Geneformer embedding table:", embedding_df.shape)
print("Scanpy PCA table:", pca_df.shape)
print("Canonical cell-type summary:", canonical_type_summary.shape)
display(pd.DataFrame([saved_metrics]).T.rename(columns={0: "value"}))


## 3. Validate dimensions, completeness, and canonical cell identity

The exported Geneformer and Scanpy PCA tables have been corrected using the canonical AnnData annotations, matched by cell barcode.

This section independently checks the exported metadata against `adata_annotated.h5ad` for all 2,671 cells before calculating silhouette scores.


In [ ]:
import anndata as ad

geneformer_features = [str(i) for i in range(256)]
pca_features = [f"PC{i}" for i in range(1, 51)]

assert len(embedding_df) == 2671, "Unexpected Geneformer cell count"
assert len(pca_df) == 2671, "Unexpected PCA cell count"

assert set(geneformer_features).issubset(embedding_df.columns)
assert set(pca_features).issubset(pca_df.columns)

assert embedding_df["cell_barcode"].is_unique
assert pca_df["cell_barcode"].is_unique

embedding_barcodes = embedding_df["cell_barcode"].astype(str)
pca_barcodes = pca_df["cell_barcode"].astype(str)

assert set(embedding_barcodes) == set(pca_barcodes), (
    "Geneformer and PCA barcode sets differ"
)

canonical_path = (
    Path.home() / "scrna-pbmc3k" / "data" / "adata_annotated.h5ad"
)
canonical_adata = ad.read_h5ad(canonical_path)

canonical_metadata = canonical_adata.obs[["cell_type", "leiden"]].copy()
canonical_metadata.index = canonical_metadata.index.astype(str)

assert len(canonical_metadata) == 2671
assert canonical_metadata.index.is_unique

canonical_aligned = canonical_metadata.reindex(embedding_barcodes)

assert canonical_aligned["cell_type"].notna().all()
assert canonical_aligned["leiden"].notna().all()

canonical_labels = canonical_aligned["cell_type"].astype(str)

pca_aligned = (
    pca_df.assign(cell_barcode=pca_barcodes)
    .set_index("cell_barcode")
    .reindex(embedding_barcodes)
    .reset_index()
)

assert (
    embedding_df["cell_type"].astype(str).to_numpy()
    == canonical_labels.to_numpy()
).all(), "Geneformer cell-type annotations differ from canonical annotations"

assert (
    pca_aligned["cell_type"].astype(str).to_numpy()
    == canonical_labels.to_numpy()
).all(), "PCA cell-type annotations differ from canonical annotations"

canonical_leiden = canonical_aligned["leiden"].astype(str)

assert (
    embedding_df["leiden"].astype(str).to_numpy()
    == canonical_leiden.to_numpy()
).all(), "Geneformer Leiden labels differ from canonical annotations"

assert (
    pca_aligned["leiden"].astype(str).to_numpy()
    == canonical_leiden.to_numpy()
).all(), "PCA Leiden labels differ from canonical annotations"

X_geneformer = embedding_df[geneformer_features].to_numpy(dtype=float)
X_pca = pca_aligned[pca_features].to_numpy(dtype=float)

assert np.isfinite(X_geneformer).all()
assert np.isfinite(X_pca).all()

assert X_geneformer.shape == (2671, 256)
assert X_pca.shape == (2671, 50)
assert canonical_labels.nunique() == 6

print("Canonical metadata validation passed.")
print("Cells:", len(canonical_labels))
print("Geneformer dimensions:", X_geneformer.shape)
print("Scanpy PCA dimensions:", X_pca.shape)
print("Canonical cell types:", canonical_labels.nunique())
print("Canonical annotation matches: 2671 / 2671")
print(canonical_labels.value_counts())


## 4. Recalculate overall silhouette metrics using canonical labels

Silhouette scores are recalculated directly from the saved embedding matrices using the corrected canonical cell-type assignments.

This avoids relying on the historical per-cell silhouette file, which was calculated using the original saved annotations.


In [ ]:
from sklearn.metrics import silhouette_score

geneformer_mean = float(
    silhouette_score(X_geneformer, canonical_labels, metric="cosine")
)
pca_mean = float(
    silhouette_score(X_pca, canonical_labels, metric="cosine")
)
difference = geneformer_mean - pca_mean

recalculated_metrics = pd.DataFrame({
    "representation": ["Geneformer V1", "Scanpy PCA"],
    "dimensions": [256, 50],
    "cosine_silhouette": [geneformer_mean, pca_mean],
})

display(recalculated_metrics.round(6))
print(f"Geneformer minus PCA: {difference:.6f}")

assert np.isclose(
    geneformer_mean,
    saved_metrics["geneformer_overall_silhouette"],
    atol=1e-8
), "Geneformer score does not match canonical metrics"

assert np.isclose(
    pca_mean,
    saved_metrics["scanpy_pca_overall_silhouette"],
    atol=1e-8
), "PCA score does not match canonical metrics"

assert np.isclose(
    difference,
    saved_metrics["difference_geneformer_minus_pca"],
    atol=1e-8
), "Difference does not match canonical metrics"

print("Canonical overall metrics independently reproduced.")


## 5. Compare mean silhouette by cell type

Positive silhouette values generally indicate that a cell is closer to its assigned group than to other groups; negative values indicate possible overlap or assignment mismatch. These values evaluate separation relative to the supplied labels, not biological correctness.


In [ ]:
from sklearn.metrics import silhouette_samples

geneformer_samples = silhouette_samples(
    X_geneformer, canonical_labels, metric="cosine"
)
pca_samples = silhouette_samples(
    X_pca, canonical_labels, metric="cosine"
)

per_cell_canonical = pd.DataFrame({
    "cell_barcode": embedding_barcodes.to_numpy(),
    "leiden": canonical_leiden.to_numpy(),
    "cell_type": canonical_labels.to_numpy(),
    "Geneformer": geneformer_samples,
    "Scanpy_PCA": pca_samples,
})

# Export canonical per-cell scores with traceable cell identifiers.
per_cell_output = RESULTS_DIR / "per_cell_silhouette_scores_canonical.csv"
per_cell_canonical.to_csv(per_cell_output, index=False)

print("Canonical per-cell silhouette scores saved:", per_cell_output)
print("Rows:", len(per_cell_canonical))
print("Unique barcodes:", per_cell_canonical["cell_barcode"].nunique())

recalculated_by_type = (
    per_cell_canonical.groupby("cell_type", as_index=False)
    .agg(
        cells=("Geneformer", "size"),
        Geneformer_mean=("Geneformer", "mean"),
        PCA_mean=("Scanpy_PCA", "mean"),
        Geneformer_median=("Geneformer", "median"),
        PCA_median=("Scanpy_PCA", "median"),
    )
)

recalculated_by_type["Difference_Geneformer_minus_PCA"] = (
    recalculated_by_type["Geneformer_mean"]
    - recalculated_by_type["PCA_mean"]
)

display(recalculated_by_type.sort_values("cell_type").round(6))

# Compare the independently recalculated values with the canonical audit CSV.
comparison = recalculated_by_type.merge(
    canonical_type_summary,
    on="cell_type",
    suffixes=("_recalculated", "_reference"),
    validate="one_to_one",
)

assert len(comparison) == 6
assert (
    comparison["cells_recalculated"]
    == comparison["cells_reference"]
).all()

for metric in [
    "Geneformer_mean",
    "PCA_mean",
    "Geneformer_median",
    "PCA_median",
    "Difference_Geneformer_minus_PCA",
]:
    assert np.allclose(
        comparison[f"{metric}_recalculated"],
        comparison[f"{metric}_reference"],
        atol=1e-8,
    ), f"Canonical summary mismatch: {metric}"

print("All six canonical cell-type summaries independently reproduced.")


## 6. Plot the saved UMAP comparison

The existing UMAP coordinates are used for visualization. Both coordinate tables contain canonical cell-type annotations matched by cell barcode.

The UMAP coordinates are not recalculated. Their orientations and coordinate scales are not directly comparable because the two representations were independently embedded.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, frame, title in [
    (axes[0], geneformer_umap.copy(), "Geneformer V1 embeddings"),
    (axes[1], pca_umap.copy(), "Scanpy PCA representation"),
]:
    required = {"UMAP1", "UMAP2", "cell_type", "cell_barcode"}
    assert required.issubset(frame.columns)
    assert len(frame) == 2671
    assert frame["cell_barcode"].is_unique

    frame["cell_barcode"] = frame["cell_barcode"].astype(str)
    expected_labels = canonical_metadata.reindex(
        frame["cell_barcode"]
    )["cell_type"]

    assert expected_labels.notna().all()
    assert (
        frame["cell_type"].astype(str).to_numpy()
        == expected_labels.astype(str).to_numpy()
    ).all(), "UMAP cell-type annotations differ from canonical annotations"

    assert np.isfinite(frame[["UMAP1", "UMAP2"]].to_numpy(dtype=float)).all()

    for cell_type, group in frame.groupby("cell_type", sort=True):
        ax.scatter(
            group["UMAP1"],
            group["UMAP2"],
            s=8,
            alpha=0.7,
            label=cell_type,
            rasterized=True
        )

    ax.set_title(title)
    ax.set_xlabel("UMAP 1")
    ax.set_ylabel("UMAP 2")

handles, labels = axes[1].get_legend_handles_labels()
fig.legend(
    handles,
    labels,
    title="Canonical cell type",
    loc="center left",
    bbox_to_anchor=(1.0, 0.5)
)
fig.tight_layout(rect=[0, 0, 0.82, 1])
plt.show()


## 7. Interpretation

Using canonical cell-type assignments, the cosine silhouette results are:

- Geneformer V1 (256 dimensions): **0.1072**
- Scanpy PCA (50 dimensions): **0.2099**
- Difference (Geneformer minus PCA): **-0.1027**

For this PBMC dataset and these configurations, Scanpy PCA separates the supplied cell-type labels more strongly under the cosine silhouette metric. This does not establish universal superiority of PCA over foundation-model embeddings.

### Annotation provenance

The canonical reference is `adata_annotated.h5ad`. The Geneformer embeddings, Scanpy PCA scores, and both UMAP coordinate tables have been corrected and independently verified against the canonical annotations by cell barcode.

All 2,671 cells now match the canonical cell-type assignments. The two discrepancies identified in the original exported tables are retained in `comparison_metrics.json` under `historical_annotation_audit`.

### Limitations

- One dataset and one Geneformer configuration were evaluated.
- The labels are preliminary annotations, not independent biological ground truth.
- Rare groups have small sample sizes, so their group-specific scores are less stable.
- The two representations have different dimensionalities and construction methods.
- UMAP is a visualization, not a quantitative measure of representation quality. Existing coordinates were retained, and canonical cell-type annotations were validated for both plots.
- This notebook validates saved embeddings and recalculates label-based metrics. It does **not** reproduce Geneformer tokenization, model inference, or raw-count-to-PCA preprocessing from scratch.
- The earlier Geneformer tokenization workflow required a pandas compatibility patch. A clean, fully reproducible environment and end-to-end pipeline remain future work.

## 8. Output files

The source outputs are in `results/verified_comparison/`. This notebook does not overwrite those files. The historical per-cell silhouette file and original saved-label summary are retained separately and are not used for the canonical calculations.
